# Phase 1 — Data Wrangling and Exploration

**Goal:** Turn the PHEME folder structure into clean, queryable dataframes and sanity-check the data before we touch any analysis.

**Outputs of this notebook** (written to `../data/processed/`):
- `threads.parquet` — one row per source tweet (the unit of analysis)
- `tweets.parquet` — one row per tweet (source + every reply)
- `edges.parquet` — parent→child reply edges, for graph construction in Phase 2

**Why this design?** Cascade analysis needs three things: (1) a thread-level table to compute per-cascade metrics, (2) a tweet-level table for user/text features, and (3) an edge list to build the reply tree. Storing them separately keeps each table tidy and lets us join only what we need later.

---

## 1. Setup

Point `PHEME_ROOT` at wherever you've unzipped the PHEME dataset. The expected layout is:

```
<PHEME_ROOT>/
  charliehebdo-all-rnr-threads/
    rumours/<thread_id>/...
    non-rumours/<thread_id>/...
  ferguson-all-rnr-threads/
  germanwings-crash-all-rnr-threads/
  ...
```

If you only have the small sample I built from the example tweet, point at `../data/sample` and you'll see one thread come through.


In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the src/ module importable
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from pheme_loader import PhemeLoader

# CHANGE THIS to the path where you've unzipped PHEME's
# 'all-rnr-annotated-threads' folder.
PHEME_ROOT = Path('/Users/henri/Documents/ETH/data_science/rumors/all-rnr-annotated-threads')   # placeholder for testing
# PHEME_ROOT = Path('../data/all-rnr-annotated-threads')   # real run

OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 2. Load everything

On the full PHEME-9 dataset this takes ~30–60 seconds and produces ~6,400 threads and ~100k tweets. The loader is in `src/pheme_loader.py` — go read it. The two non-obvious things it handles for you:

1. **Veracity normalization.** PHEME annotation files are inconsistent across events (some use `category: "true"`, some use `"true": "1"`, the older PHEME-5 uses `misinformation: 0/1`). The loader normalizes all of them into a single `veracity` column with values `{true, false, unverified, nonrumour}`.
2. **Reply tree from `structure.json`.** This file is the authoritative tree topology. We trust it over `in_reply_to_status_id` because the structure is sometimes more complete (and `in_reply_to_status_id` can point to a tweet that's been deleted).

In [2]:
loader = PhemeLoader(PHEME_ROOT)
threads_df, tweets_df, edges_df = loader.load_all()

print(f'threads_df: {len(threads_df):>7,} rows')
print(f'tweets_df:  {len(tweets_df):>7,} rows')
print(f'edges_df:   {len(edges_df):>7,} rows')

Parsed 6425 threads, skipped 0 (malformed/missing).
threads_df:   6,425 rows
tweets_df:  105,354 rows
edges_df:    95,076 rows


## 3. Sanity checks

Before any analysis: are the things we'll rely on actually there?

In [3]:
# 3a. Class distribution — overall and per event
print('Overall:')
print(threads_df['veracity'].value_counts())
print()
print('Per event:')
print(
    threads_df.groupby('event')['veracity']
    .value_counts()
    .unstack(fill_value=0)
)

Overall:
veracity
nonrumour                                                                     4022
prince will play a show at massey hall in toronto on november 4                131
there is a hostage situation at a cafe in sydney                                89
m. brown was involved in a robbery before being shot                            87
prince will play a show in toronto on november 4                                73
                                                                              ... 
six more hostages have escaped from the cafe (after initial 5)                   1
at least 7 hostages have been released from cafe (after initial 5 ran out)       1
at least four people have been injured                                           1
at least 12 more hostages have emerged from cafe (after initial 5)               1
a bomb detection robot has entered the cafe                                      1
Name: count, Length: 250, dtype: int64

Per event:
veracity          

**What to look for here:** PHEME is heavily imbalanced — non-rumours dominate every event, and within rumours, `unverified` is usually the largest class. This is real signal, not a bug: most of what circulated during these breaking news events was unverified. But it means you'll need to (a) report stratified results and (b) be careful with statistical tests that assume balanced groups.

In [4]:
# 3b. How complete are the reply trees?
# n_tweets_in_structure = number of tweets the structure tree references
# n_reactions + 1     = number of tweets we actually have JSON for (source + reactions)
threads_df['tweets_we_have'] = threads_df['n_reactions'] + 1
threads_df['structure_completeness'] = (
    threads_df['tweets_we_have'] / threads_df['n_tweets_in_structure']
).clip(upper=1.0)

print('Structure completeness distribution:')
print(threads_df['structure_completeness'].describe())
print()
print('Threads where structure references tweets we don\'t have:',
      (threads_df['structure_completeness'] < 1.0).sum(),
      f'({(threads_df["structure_completeness"] < 1.0).mean():.1%})')

Structure completeness distribution:
count    6425.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: structure_completeness, dtype: float64

Threads where structure references tweets we don't have: 0 (0.0%)


**Why this matters for the presentation:** when we report cascade size, we'll use `n_tweets_in_structure` (the tree topology), not `n_reactions` (the JSONs we have). The tree is the canonical record of what spread; the missing JSONs just mean we lack text/user features for those nodes. Be honest about this in the methodology slide.

If completeness is below 80% on average, mention it as a limitation. If it's above 95%, you can essentially ignore it.

In [5]:
# 3c. Are the timestamps parseable?
n_bad_source = threads_df['source_created_at'].isna().sum()
n_bad_tweet = tweets_df['created_at'].isna().sum()
print(f'Source tweets with unparseable timestamp: {n_bad_source}')
print(f'Reply tweets with unparseable timestamp:  {n_bad_tweet}')

# Time span per event — useful sanity check
if len(threads_df) > 1:
    print()
    print('Time span per event:')
    print(threads_df.groupby('event')['source_created_at'].agg(['min', 'max', 'count']))

Source tweets with unparseable timestamp: 0
Reply tweets with unparseable timestamp:  0

Time span per event:
                                        min                       max  count
event                                                                       
charliehebdo      2015-01-07 11:06:08+00:00 2015-01-09 16:40:39+00:00   2079
ebola-essien      2014-10-12 14:44:23+00:00 2014-10-15 07:23:48+00:00     14
ferguson          2014-08-09 22:33:06+00:00 2014-08-15 23:57:45+00:00   1143
germanwings-crash 2015-03-24 10:37:41+00:00 2015-03-27 20:17:38+00:00    469
gurlitt           2014-11-20 01:20:18+00:00 2014-11-24 11:49:01+00:00    138
ottawashooting    2014-10-22 13:55:50+00:00 2014-10-22 23:55:12+00:00    890
prince-toronto    2014-11-03 22:29:15+00:00 2014-11-05 03:00:34+00:00    233
putinmissing      2015-03-13 00:04:02+00:00 2015-03-16 05:36:16+00:00    238
sydneysiege       2014-12-14 23:02:38+00:00 2014-12-15 15:58:44+00:00   1221


In [6]:
bad_children = edges_df[~edges_df['child_id'].str.isdigit()]
bad_parents = edges_df[~edges_df['parent_id'].str.isdigit()]

print(f"Bad child_ids:  {len(bad_children)} edges")
print(f"Bad parent_ids: {len(bad_parents)} edges")
print()
print("Unique bad child_id values (top 20):")
print(bad_children['child_id'].value_counts().head(20))
print()
print("Threads affected:")
print(bad_children['thread_id'].value_counts().head(10))

Bad child_ids:  0 edges
Bad parent_ids: 0 edges

Unique bad child_id values (top 20):
Series([], Name: count, dtype: int64)

Threads affected:
Series([], Name: count, dtype: int64)


In [7]:
# 3d. Do the edge IDs make sense? Every parent_id and child_id should be a tweet
# id (string of digits). Every thread_id in edges should appear in threads_df.
assert edges_df['thread_id'].isin(threads_df['thread_id']).all(), 'orphan edges!'
assert edges_df['parent_id'].str.isdigit().all(), 'non-numeric parent ids'
assert edges_df['child_id'].str.isdigit().all(), 'non-numeric child ids'
print('Edge integrity checks passed.')

Edge integrity checks passed.


## 4. First look at the spread question

We haven't computed any real metrics yet — that's Phase 2. But `n_tweets_in_structure` is a free first-pass cascade size, and we can already see whether it differs by veracity.

In [8]:
summary = threads_df.groupby('veracity')['n_tweets_in_structure'].agg(
    n='count', mean='mean', median='median', p90=lambda s: s.quantile(0.9), max='max'
).round(2)
print('Cascade size (tweets in reply tree) by veracity:')
print(summary)

Cascade size (tweets in reply tree) by veracity:
                                                     n   mean  median   p90  max
veracity                                                                        
" multiple"  gunmen were involved in the charli...   3  10.67     8.0  18.4   21
"several" more hostages have escaped from the c...   7  16.43    16.0  19.6   22
(alleged) isis militants are behind the hostage...   8  21.50    20.5  31.3   32
(at least) 10 people are dead at charlie hebdo ...  23  12.70     9.0  23.0   61
(up to) 150 people perished in the crash (144 p...  12   8.33     6.5  15.6   24
...                                                 ..    ...     ...   ...  ...
up to five more hostages have escaped from the ...   4   7.25     8.0  11.4   12
witnesses have cellphone video of the mike brow...   1   1.00     1.0   1.0    1
woman shot in drive-by shooting shot video of p...   3  10.00     3.0  20.6   25
wristbands for the prince show will be availabl...   5   4.8

**A note on means vs. medians here.** Cascade sizes are heavy-tailed — a handful of mega-cascades will pull the mean way above the median. Always report both, and on visualizations always use log scale. The median tells you about typical behavior; the mean (and max, p90, p99) tells you about the tail, which is often where the action is for rumors.

If you see in this table that, say, false rumors have a higher median *and* higher tail than non-rumours, that's already an early hint of the spread asymmetry — something to come back to in the final story.

## 5. Save processed dataframes

Parquet (not CSV) because it preserves dtypes (especially the timezone-aware datetimes) and is ~10x smaller and faster to reload. If your group doesn't have `pyarrow` installed, swap to `to_pickle`.

In [9]:
# Try parquet first (preserves dtypes, smaller files); fall back to pickle.
try:
    threads_df.to_parquet(OUTPUT_DIR / 'threads.parquet', index=False)
    tweets_df.to_parquet(OUTPUT_DIR / 'tweets.parquet', index=False)
    edges_df.to_parquet(OUTPUT_DIR / 'edges.parquet', index=False)
    ext = 'parquet'
except ImportError:
    print('pyarrow not installed — falling back to pickle. (pip install pyarrow for smaller files.)')
    threads_df.to_pickle(OUTPUT_DIR / 'threads.pkl')
    tweets_df.to_pickle(OUTPUT_DIR / 'tweets.pkl')
    edges_df.to_pickle(OUTPUT_DIR / 'edges.pkl')
    ext = 'pkl'

print('Saved to', OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.glob(f'*.{ext}')):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')

Saved to /Users/henri/Documents/ETH/data_science/data/processed
  edges.parquet  (1966.4 KB)
  threads.parquet  (752.4 KB)
  tweets.parquet  (12059.3 KB)


---

## What we have now, and what's next

**We have:**
- `threads_df` — one row per source tweet, with veracity label, event, source-user features, and a first-pass cascade size.
- `tweets_df` — every tweet (source + replies) with timestamps and user features.
- `edges_df` — the reply-tree edge list, ready to feed into NetworkX.
- A clear-eyed picture of how complete the data actually is.

**Phase 2 will compute the real metrics** on top of these dataframes:
- *Speed:* time-to-first-reply, time-to-half-cascade, reply velocity by hour
- *Reach:* total cascade size, unique users, distinct branches
- *Structure:* tree depth, max breadth, structural virality (Goel et al.)

Then Phase 3 compares those metrics across veracity classes with proper non-parametric tests, and Phase 4 builds the six figures that carry the presentation.
